# Patch-Size Sweep + PatchCleanser Baseline

In [1]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
import sys
sys.path.insert(0, os.path.abspath('.'))

assert os.path.isdir('src'), f"src/ not found in {_nb_dir}"

from src.config import DEVICE, RESULTS_DIR, CHECKPOINT_DIR
from src.model import get_model
from src.data_loader import build_partition_indices, get_partition
from src.attack import load_patch, apply_patch
from src.gradcam_defense import GradCAM
from src.feature_squeeze import load_threshold

print(f"Device: {DEVICE}")
model        = get_model()
patch        = load_patch()
partitions   = build_partition_indices()
gradcam      = GradCAM(model, target_layer_name='layer4')
fs_threshold = load_threshold()
print(f"FS threshold: {fs_threshold:.3f}")

import torch
results = torch.load('results/results_with_ci.pt', weights_only=False)
print("Base results loaded:", list(results.keys()))


[config] Using device: mps
Device: mps
Loading pretrained ResNet-50 …
Ready
Patch loaded  shape: (3, 89, 89)
Threshold loaded: 0.150
FS threshold: 0.150
Base results loaded: ['no_defense', 'jpeg', 'feature_squeeze', 'gradcam_only', 'hybrid']


## Part A: 44×44 ablation (should match nb1 result)

In [2]:

import os, torch
_path = 'results/patches/patch_44px.pt'
if os.path.exists(_path):
    print("[SKIP] patch_44px.pt exists.")
    patch_44 = torch.load(_path, weights_only=False)
    from src.attack import evaluate_attack
    from src.data_loader import get_test_subset
    imgs_44, lbls_44 = get_test_subset(n=200, seed=77)
    r44 = evaluate_attack(model, patch_44, imgs_44, lbls_44)
    print(f"44px ASR: {r44['attack_sr']:.1f}%")
else:
    print("patch_44px.pt not found -- run nb1 first (Step 10).")


[SKIP] patch_44px.pt exists.
Clean acc: 100.0%  Attack SR: 44.0%  Target rate: 53.5%
44px ASR: 44.0%


## Part B: Full patch-size sweep (44 / 60 / 75 / 89 / 110 px)



In [3]:
import os, sys, torch
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
sys.path.insert(0, os.path.abspath('.'))

# Lazy init guards so this cell works after a kernel restart
from src.data_loader import build_partition_indices, get_partition
from src.model import get_model
from src.attack import load_patch, apply_patch
from src.gradcam_defense import GradCAM
from src.feature_squeeze import load_threshold
if 'partitions' not in dir(): partitions = build_partition_indices()
if 'model'      not in dir(): model      = get_model()
if 'patch'      not in dir(): patch      = load_patch()
if 'gradcam'    not in dir(): gradcam    = GradCAM(model, target_layer_name='layer4')
if 'fs_threshold' not in dir(): fs_threshold = load_threshold()

_path = 'results/patch_size_sweep.pt'

# Stale detection: the old bug made EVERY patch size use EXACTLY the same
# patch (89px), so the raw (unrounded) stage1_fpr values are bit-for-bit
# identical across all sizes. Only delete on exact equality, not rounding.
# Also honour a '_valid' flag written by new runs.
if os.path.exists(_path):
    _sw = torch.load(_path, weights_only=False)
    _is_valid = _sw.get('_valid', False)
    if not _is_valid:
        _raw_fprs = [_sw[px]['stage1_fpr'] for px in sorted(k for k in _sw if k != '_valid')]
        _all_identical = len(set(_raw_fprs)) == 1
        if _all_identical:
            print(f"[STALE] Raw Stage-1 FPRs all identical ({_raw_fprs[0]}) "
                  f"-- cached with apply_patch bug. Deleting.")
            os.remove(_path)
        else:
            print(f"[INFO] No _valid flag but FPRs differ {_raw_fprs} -- keeping.")

if os.path.exists(_path):
    print("[SKIP] patch_size_sweep.pt exists and is valid -- loading.")
    from src.patch_size_sweep import print_sweep_table
    sweep_results = torch.load(_path, weights_only=False)
    # strip internal keys before printing
    sweep_results_clean = {k: v for k, v in sweep_results.items() if k != '_valid'}
    print_sweep_table(sweep_results_clean)
    sweep_results = sweep_results_clean
else:
    print("[RUN] Running patch-size sweep...")
    from src.patch_size_sweep import run_sweep, print_sweep_table
    from src.data_loader import get_loaders
    train_loader_sw, _, _ = get_loaders()
    sweep_results = run_sweep(train_loader_sw, save=True)
    print_sweep_table(sweep_results)


[SKIP] patch_size_sweep.pt exists and is valid -- loading.

  Size   Area      ASR   S1 Det   S1 FPR    Recov      FPR   ms/img
----------------------------------------------------------------------------------------
   44    3.9%    43.3%    49.0%    34.3%    79.3%     0.7%    40.4
   60    7.2%    71.5%    72.7%    34.3%    78.3%     0.7%    50.5
   75   11.2%    73.2%    87.3%    34.3%    89.0%     0.7%    57.3
   89 *  15.8%    90.3%    92.0%    34.3%    90.0%     0.7%    59.6
  110   24.1%    90.6%    90.0%    34.3%    62.7%     0.7%    59.8
* Primary evaluation patch size


## Part C: PatchCleanser double-masking baseline


In [ ]:

import os, sys, torch
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
sys.path.insert(0, os.path.abspath('.'))

from src.data_loader import build_partition_indices, get_partition
from src.attack import load_patch, apply_patch
from src.model import get_model
from src.gradcam_defense import GradCAM
from src.feature_squeeze import load_threshold

if 'partitions' not in dir():
    partitions = build_partition_indices()
if 'model' not in dir():
    model = get_model()
if 'patch' not in dir():
    patch = load_patch()
if 'gradcam' not in dir():
    gradcam = GradCAM(model, target_layer_name='layer4')
if 'fs_threshold' not in dir():
    fs_threshold = load_threshold()
if 'results' not in dir():
    results = torch.load('results/results_with_ci.pt', weights_only=False)

images_clean, labels = get_partition("final_eval", partitions)
images_patched, _    = apply_patch(images_clean.clone(), patch, augment=False)
print(f"Loaded {images_clean.size(0)} images for PatchCleanser")


Loaded 1000 images for PatchCleanser


In [ ]:
import os, torch

_path = 'results/results_with_patchcleanser.pt'

if os.path.exists(_path):
    _r = torch.load(_path, weights_only=False)
    _pc = _r.get('patchcleanser', {})
    _note = _pc.get('note', '')
    _n = int(_pc.get('n_images', 0))
    if _n == 0 or _n < 500:  # _n==0 means old format without n_images key
        print(f"[STALE] Cached PatchCleanser result has n={_n} (<500).")
        print("Deleting stale cache and re-running on full final_eval (1000 images)...")
        os.remove(_path)
    elif 'n=' not in _note and _n == 0:
        print("[STALE] Cached result has no n_images field (old format).")
        print("Deleting stale cache and re-running...")
        os.remove(_path)

if os.path.exists(_path):
    print("[SKIP] results_with_patchcleanser.pt exists and looks valid -- loading.")
    from src.patchcleanser_baseline import PATCHCLEANSER_LITERATURE
    results_extended = torch.load(_path, weights_only=False)
    if "patchcleanser" in results_extended:
        pc = results_extended["patchcleanser"]
        print(f"  n_images:      {pc.get('n_images', 'unknown')}")
        print(f"  Clean acc:     {pc['clean_acc']:.1f}%")
        print(f"  Recovered acc: {pc['recovered_acc']:.1f}%")
        print(f"  FPR:           {pc['fpr']:.1f}%")
else:
    print("[RUN] Running PatchCleanser on final_eval (1000 images)...")
    from src.patchcleanser_baseline import (
        evaluate_patchcleanser, add_patchcleanser_to_table, PATCHCLEANSER_LITERATURE
    )
    pc_result = evaluate_patchcleanser(
        model, images_clean, images_patched, labels,
        patch_size_px=89, k=2,
    )
    results_extended = add_patchcleanser_to_table(dict(results), pc_result)
    torch.save(results_extended, _path)
    print(f"Saved. n_images={pc_result.get('n_images', 'unknown')}")

print(f"\nLiterature (Xiang et al. 2022):")
print(f"  Clean acc: {PATCHCLEANSER_LITERATURE['clean_acc']}%")
print(f"  Certified: {PATCHCLEANSER_LITERATURE['certified_acc']}% at "
      f"{PATCHCLEANSER_LITERATURE['patch_size_pct']}% patch area")
print(f"  Note: {PATCHCLEANSER_LITERATURE['note']}")


[RUN] Running PatchCleanser on final_eval (1000 images)...

PatchCleanser evaluation: k=2, 4 single masks, 6 double masks, patch_size=89px


  PatchCleanser (patched): 100%|██████████| 1000/1000 [01:45<00:00,  9.44it/s]



PatchCleanser (k=2) results:
  Clean accuracy:          99.6%
  Recovered accuracy:      65.1%
  False positive rate:     0.4%
  Certified (clean):       98.1%
  Certified (patched):     17.6%
  Forward passes/image:    10
Saved. n_images=1000

Literature (Xiang et al. 2022):
  Clean acc: 99.1%
  Certified: 49.8% at 2.0% patch area
  Note: Their certified accuracy is for 2% patch area -- directly comparable to our 4% (44px) setting, not our primary 16% (89px) setting. Ours is a harder attack; certified defenses degrade sharply as patch size grows.


## Summary

In [6]:

import torch
from src.evaluate import print_results_with_ci

print("=== PATCH-SIZE SWEEP ===")
try:
    from src.patch_size_sweep import print_sweep_table
    sweep_results = torch.load('results/patch_size_sweep.pt', weights_only=False)
    print_sweep_table(sweep_results)
except FileNotFoundError:
    print("Not yet complete.")

print("\n=== TABLE II WITH PATCHCLEANSER ===")
try:
    r = torch.load('results/results_with_patchcleanser.pt', weights_only=False)
    print_results_with_ci(r)
except FileNotFoundError:
    print("Not yet complete.")


=== PATCH-SIZE SWEEP ===

  Size   Area      ASR   S1 Det   S1 FPR    Recov      FPR   ms/img
----------------------------------------------------------------------------------------
   44    3.9%    43.3%    49.0%    34.3%    79.3%     0.7%    40.4
   60    7.2%    71.5%    72.7%    34.3%    78.3%     0.7%    50.5
   75   11.2%    73.2%    87.3%    34.3%    89.0%     0.7%    57.3
   89 *  15.8%    90.3%    92.0%    34.3%    90.0%     0.7%    59.6
  110   24.1%    90.6%    90.0%    34.3%    62.7%     0.7%    59.8
* Primary evaluation patch size

=== TABLE II WITH PATCHCLEANSER ===

Method                       Clean acc       Attack SR       Recovered          FPR
-------------------------------------------------------------------------------------
no_defense             99.8% [99.5-100.0] 84.3% [82.1-86.4] 15.7% [13.6-17.9] 0.0% [0.0-0.0]
jpeg                   99.8% [99.5-100.0] 46.1% [43.0-49.4] 53.9% [50.6-57.0] 0.2% [0.0-0.5]
feature_squeeze        90.6% [88.8-92.3] 21.2% [18.7-23